# The dimension of `supp rho_L` on a lattice patch

## The objects, named

* A **Gaussian state** of Majorana modes $\gamma_a$ (Hermitian, $\gamma_a^2 = 1$,
  $\gamma_a\gamma_b = -\gamma_b\gamma_a$) is fixed by its **covariance**
  $\Gamma_{ab} = \tfrac{i}{2}\langle[\gamma_a,\gamma_b]\rangle$, a real antisymmetric matrix.
  The code labels the Majoranas by integers, and site $r$ of a lattice owns the labels $2r$
  and $2r+1$ everywhere below.
* A **patch** $L$ is a set of sites; keeping modes $\in L$ and tracing out everything else $\bar{L}$ leaves the  **patch state** $\rho_L=\mathrm{\mathop{tr}}_{\bar{L}}\ket{\Gamma}\bra{\Gamma}$, whose **block** $\Gamma_L$ is the submatrix of $\Gamma$ on the
  patch's Majoranas.
* The **Williamson decomposition** of the block writes
  $ \Gamma_L = \bigoplus_{k}R^{T}\mathrm{diag}(\lambda_k J)R $ with
  $J = \begin{bmatrix}0&-1\\1&0\end{bmatrix}$, $R$ orthogonal and $0 \leq \lambda_k \leq 1$.
  Mode $k$ is **pure** when $\lambda_k = 1$ and **mixed** otherwise.  A pure mode contributes
  one dimension to $\mathrm{supp}\,\rho_L$ and a mixed mode two, so with a **tolerance**
  $\epsilon$: 
  $$\dim \mathrm{supp}\,\rho_L = 2^{\#\{\lambda_k < 1 - \epsilon\}}.$$
  The exponent $\#\{\lambda_k < 1 - \epsilon\}$ is what this notebook measures.
* $S$ is the von Neumann entropy of the patch state, $-\mathrm{Tr}\,\rho_L\ln\rho_L$
  (`linalg.entropy`, with $k_B=1$).

## What are the pure states made of

For a pure global state $\Gamma = \begin{pmatrix}\Gamma_L & C\\- C^T&\Gamma_{\bar{L}} \end{pmatrix}$, block matrix arguments gives 

$$\Gamma_L^2 + 1 = C\,C^{T},$$

so $\#\{\lambda_k < 1\} =\operatorname{rank} C$ exactly, and $C$ -- the **cross
covariance** between the patch and everything outside it -- is small exactly where the patch
couples weakly to its outside, i.e. away from the boundary of the patch.  The rank is
therefore maximal in exact arithmetic; what a finite tolerance resolves is a layer at the
boundary, of a depth this notebook measures against $\epsilon$ rather than assumes.


In [ ]:
import math

import matplotlib.pyplot as plt
import torch
from torch import Tensor

from FreeFermion.GfPEPS import GfPEPS
from FreeFermion.geometry import Boundary, Geometry, HONEYCOMB, Lattice
from FreeFermion.linalg import entropy, occupations
from base import REAL

TOLERANCES = (1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12)
QUOTED = 1e-8          # the tolerance every count in the tables is quoted at


def mixed_count(covariance: Tensor, tolerance: float = QUOTED) -> int:
    r'''
    The count of a block at one tolerance: the number of modes with $\lambda_k < 1 - \epsilon$, which are the occupations $p_k = (1-\lambda_k)/2$ above $\epsilon/2$.
    Args:
        covariance: real antisymmetric covariance of the patch, shape (2n, 2n).
        tolerance: the tolerance on $1 - \lambda_k$.
    Returns:
        The count.
    '''
    return int((occupations(covariance) > tolerance / 2).sum())


def num_pure(covariance: Tensor,
              tolerances: tuple[float, ...] = TOLERANCES) -> dict[float, int]:
    '''
    The number of pure modes of a block at every tolerance, keyed by the tolerance, for the tables and the depth plot that need the whole curve.
    Args:
        covariance: real antisymmetric covariance of the patch.
        tolerances: the tolerances.
    Returns:
        tolerance -> count.
    '''
    p = occupations(covariance)
    return {tolerance: int((p > tolerance / 2).sum()) for tolerance in tolerances}


def patch_covariance(covariance: Tensor, patch: list[int]) -> Tensor:
    '''
    The block of a patch, site r owning the Majoranas 2r and 2r+1.
    Args:
        covariance: real antisymmetric covariance of the whole state.
        patch: site labels of the patch.
    Returns:
        The block, shape (2 * len(patch), 2 * len(patch)).
    '''
    majoranas = sorted(2 * site + odd for site in patch for odd in (0, 1))
    return covariance[majoranas][:, majoranas]

## 1. One dimension: a block of a ring

The Kitaev chain on a ring of $N$ sites, site $j$ owning the Majoranas $2j$ and $2j+1$,
$$H = -w\sum_j\left(c_j^\dagger c_{j+1} + \mathrm{h.c.}\right)
     -\mu\sum_j\left(c_j^\dagger c_j - \tfrac12\right)
     +\Delta\sum_j\left(c_j c_{j+1} + \mathrm{h.c.}\right).$$
`kitaev_terms` writes the terms of that Hamiltonian, `majorana_matrix` collects them into the
real antisymmetric matrix $M$ of $H = (i/4)\sum_{ab}M_{ab}\gamma_a\gamma_b$, and
`GfPEPS.from_hamiltonian` returns the ground state covariance of $M$ through its sign
function.  The patch is a contiguous block of $L$ sites, the first $L$ of the ring.  `flux`
flips the sign of the wrap bond, which puts the ground state in the antiperiodic sector: the
critical Ising ring, against the gapped rings of the other parameters.

In [ ]:
def majorana_matrix(terms: list[tuple[int, int, complex]], num_majoranas: int,
                    device: torch.device = torch.device('cuda')) -> Tensor:
    r'''
    Assemble the real antisymmetric $M$ of $H = (i/4)\gamma^T M \gamma$ from the terms
    $\alpha\,\gamma_a\gamma_b$, with every $\alpha$ purely imaginary.
    Args:
        terms: (a, b, alpha) contributions, summed over repeated pairs.
        num_majoranas: dimension of M.
        device: device of the returned matrix.
    Returns:
        The real antisymmetric matrix M.
    Raises:
        ValueError: if a pair repeats a Majorana, or an alpha is not purely imaginary.
    '''
    entries: dict[tuple[int, int], float] = {}
    for a, b, alpha in terms:
        if a == b:
            raise ValueError(f'the pair ({a}, {b}) repeats a Majorana')
        if abs(alpha.real) > 1e-15:
            raise ValueError(f'alpha {alpha} of ({a}, {b}) is not purely imaginary')
        if a > b:
            a, b, alpha = b, a, -alpha
        coefficient = -2j * alpha
        if abs(coefficient.imag) > 1e-15:
            raise ValueError(f'alpha {alpha} of ({a}, {b}) gives a complex M entry')
        entries[(a, b)] = entries.get((a, b), 0.0) + coefficient.real
    matrix = torch.zeros((num_majoranas, num_majoranas), dtype=REAL, device=device)
    index = torch.tensor(list(entries), dtype=torch.long, device=device).reshape(-1, 2)
    values = torch.tensor(list(entries.values()), dtype=REAL, device=device)
    matrix[index[:, 0], index[:, 1]] = values
    return matrix - matrix.T


def kitaev_terms(num_sites: int, w: float = 1.0, delta: float = 1.0, mu: float = 0.0,
                 flux: bool = False) -> list[tuple[int, int, complex]]:
    r'''
    Terms of the Kitaev chain on a ring, site j on the Majoranas (2j, 2j+1).
    Args:
        num_sites: number of sites.
        w: hopping.
        delta: pairing.
        mu: chemical potential.
        flux: negate the wrap bond, i.e. antiperiodic boundary conditions.
    Returns:
        The (a, b, alpha) terms.
    '''
    terms = []
    for j in range(num_sites):
        jp = (j + 1) % num_sites
        sign = -1.0 if flux and j == num_sites - 1 else 1.0
        terms += [(2 * j, 2 * j + 1, -0.5j * mu),
                  (2 * j, 2 * jp + 1, sign * (-0.5j * w)),
                  (2 * jp, 2 * j + 1, sign * (-0.5j * w)),
                  (2 * j, 2 * jp + 1, sign * (0.5j * delta)),
                  (2 * jp, 2 * j + 1, sign * (-0.5j * delta))]
    return terms


NUM_SITES = 600
LENGTHS = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 200]
RINGS = {r'\mu = 1': (1.0, False), r'\mu = 1.9': (1.9, False),
         r'\mu = 3.0': (3.0, False), r'\mu = 2.0,  critical, antiperiodic': (2.0, True)}
blocks:dict[str,dict[int,Tensor]] = {}
for label, (mu, flux) in RINGS.items():
    covariance = GfPEPS.from_hamiltonian(
        majorana_matrix(kitaev_terms(NUM_SITES, mu=mu, flux=flux), 2 * NUM_SITES)).tensors[0]
    blocks[label] = {length: patch_covariance(covariance, list(range(length)))
                     for length in LENGTHS}

print(f'{"ring":>26} {"S(200)":>8} '
      + ' '.join(f'{t:>6.0e}' for t in TOLERANCES[1::2]))
for label, curves in blocks.items():
    print(f'{label:>26} {entropy(curves[200]):>8.4f} '
          + ' '.join(f'{num_pure(curves[200])[t]:>6}' for t in TOLERANCES[1::2]))

figure, axes = plt.subplots(1, 2, figsize=(9.6, 3.6))
for label, curves in blocks.items():
    axes[0].plot(LENGTHS, [mixed_count(curves[length]) for length in LENGTHS],
                 marker='.', label=rf'${label}$')
    axes[1].plot(LENGTHS, [entropy(curves[length]) for length in LENGTHS],
                 marker='.', label=rf'${label}$')
axes[0].plot([1, 200], [1, 200], 'k--', lw=1, label=r'$\#\mathrm{mixed} = L$')
axes[0].set(xscale='log', yscale='log', xlabel='$L$',
            ylabel=r'$\#\{\lambda_k < 1 - 10^{-8}\}$',
            title=r'the resolvable $\log_2\dim\mathrm{Im}\,\rho_L$')
axes[1].set(xscale='log', xlabel='$L$', ylabel='$S$', title='entropy')
for axis in axes:
    axis.legend(fontsize=7)
figure.tight_layout()
plt.show()

## 2. One dimension: the SSH chain, open and closed

The SSH chain has two sites per cell, $A$ and $B$, with $v$ inside a cell and $w$ between
cells,
$$H = -v\sum_j\left(c_{j,A}^\dagger c_{j,B} + \mathrm{h.c.}\right)
     -w\sum_j\left(c_{j,B}^\dagger c_{j+1,A} + \mathrm{h.c.}\right).$$
It is topological for $|v| < |w|$: the open chain then carries one zero mode per end, while
the ring has a gap of $2|v - w|$ and no edge state at all.  The two phases share the same
bulk spectrum, so only the boundary tells them apart, which is what a patch sees.

The chain is written as a `Lattice` with two basis sites and the two bonds of the infinite
chain; `rectangle` opens it and `torus` closes it.  The ideal chain has *exact* zero modes,
so its ground state is degenerate and $\Gamma = i\,\mathrm{sign}(iM)$ is not defined: a
Semenoff mass $\eta$ on the sublattices breaks the chiral symmetry, splits the pair by
$\eta$, and makes the ground state unique, which is what `from_hamiltonian` needs.


In [ ]:
SSH = Lattice(name='ssh',
              vectors=torch.tensor([[1.0, 0.0], [0.0, 1.0]], dtype=REAL),
              basis=torch.tensor([[0.0, 0.0], [0.5, 0.0]], dtype=REAL),
              bonds=((0, 1, (0, 0)), (1, 0, (1, 0))))


def hopping(j: int, k: int, amplitude: complex) -> list[tuple[int, int, complex]]:
    r'''
    The Majorana terms of $-\alpha\,c_j^\dagger c_k + \mathrm{h.c.}$.
    Args:
        j: site the electron leaves.
        k: site it arrives at.
        amplitude: $\alpha$, real for a hopping and $t e^{i\varphi}$ for a bond with flux.
    Returns:
        The (a, b, alpha) terms.
    '''
    return [(2 * j, 2 * k, -0.5j * amplitude.imag),
            (2 * j + 1, 2 * k + 1, -0.5j * amplitude.imag),
            (2 * j, 2 * k + 1, -0.5j * amplitude.real),
            (2 * j + 1, 2 * k, 0.5j * amplitude.real)]


def positive_spectrum(matrix: Tensor) -> Tensor:
    r'''
    The positive single-particle energies of $H = (i/4)\gamma^T M\gamma$.
    Args:
        matrix: the real antisymmetric M, shape (2n, 2n).
    Returns:
        The $n$ positive eigenvalues, ascending, on the host.
    '''
    values = torch.linalg.eigvalsh(1j * matrix.to(torch.complex128))
    return values[values > 0].cpu()


def ssh_terms(geometry: Geometry, v: float, w: float,
              eta: float) -> list[tuple[int, int, complex]]:
    r'''
    $v$ inside a cell, $w$ between cells, and a Semenoff mass $\eta$ on the sublattices.
    Args:
        geometry: the chain, whose bonds are the nearest-neighbour ones.
        v: hopping inside a cell.
        w: hopping between cells.
        eta: on-site energy $+\eta$ on sublattice A and $-\eta$ on B.
    Returns:
        The (a, b, alpha) terms.
    '''
    terms = []
    for r, rp in geometry.bonds:
        inside = bool(torch.equal(geometry.cell[r], geometry.cell[rp]))
        terms += hopping(r, rp, complex(v if inside else w))
    for r, basis in enumerate(geometry.basis):
        terms.append((2 * r, 2 * r + 1, 0.5j * eta * (1.0 if basis == 0 else -1.0)))
    return terms


NUM_CELLS, ETA = 200, 1e-3
CHAINS = {'topological, open': (0.5, 1.0, Boundary.OPEN),
          'trivial, open': (1.0, 0.5, Boundary.OPEN),
          'topological, ring': (0.5, 1.0, Boundary.PERIODIC),
          'trivial, ring': (1.0, 0.5, Boundary.PERIODIC)}
chains, chain_spectra = {}, {}
for label, (v, w, boundary) in CHAINS.items():
    geometry = Geometry.build(SSH, NUM_CELLS, 1, (boundary, Boundary.OPEN))
    matrix = majorana_matrix(ssh_terms(geometry, v, w, ETA), 2 * geometry.sites)
    covariance = GfPEPS.from_hamiltonian(matrix).tensors[0]
    chains[label] = {length: patch_covariance(covariance, list(range(length)))
                     for length in LENGTHS}
    chain_spectra[label] = positive_spectrum(matrix)

print(f'{"SSH chain":>18} {"min eps":>10} {"S(200)":>8} {"#mixed(200)":>12}')
for label, curves in chains.items():
    print(f'{label:>18} {float(chain_spectra[label].min()):>10.2e} '
          f'{entropy(curves[200]):>8.4f} {mixed_count(curves[200]):>12}')

figure, axes = plt.subplots(1, 3, figsize=(13.5, 3.4))
for label, values in chain_spectra.items():
    axes[0].plot(range(1, len(values) + 1), values, '.', ms=3, label=label)
axes[0].set(xlabel='state index', ylabel=r'$\varepsilon_k$', yscale='log',
            ylim=(3e-4, 4), title='SSH: single-particle spectrum')
for label, curves in chains.items():
    axes[1].plot(LENGTHS, [mixed_count(curves[length]) for length in LENGTHS],
                 marker='.', label=label)
    axes[2].plot(LENGTHS, [entropy(curves[length]) for length in LENGTHS],
                 marker='.', label=label)
axes[1].plot([1, 200], [1, 200], 'k--', lw=1, label=r'$\#\mathrm{mixed} = L$')
axes[1].set(xscale='log', yscale='log', xlabel='$L$',
            ylabel=r'$\#\{\lambda_k < 1 - 10^{-8}\}$', title='SSH: resolvable rank')
axes[2].set(xscale='log', xlabel='$L$', ylabel='$S$', title='SSH: entropy')
for axis in axes:
    axis.legend(fontsize=7)
figure.tight_layout()
plt.show()


## 3. Two dimensions: the Haldane model, open and closed

The Haldane model is graphene with a second-neighbour flux and a sublattice mass,
$$H = -t\sum_{\langle ij\rangle}\left(c_i^\dagger c_j + \mathrm{h.c.}\right)
     -t_2\sum_{\langle\langle ij\rangle\rangle}\left(e^{i\varphi_{ij}}c_i^\dagger c_j + \mathrm{h.c.}\right)
     + M\sum_i \xi_i\, c_i^\dagger c_i,$$
with $\xi = +1$ on $A$ and $-1$ on $B$, and the phase $\pm\varphi$ fixed by the sublattice.
It is a Chern insulator for $|M| < 3\sqrt3\,t_2|\sin\varphi|$: on the open sheet the boundary
carries a chiral mode inside the gap, and on the torus there is none.  That threshold is the
check on the phase convention, and the sweep below finds it.  It needs a sheet whose cell
count is a multiple of 3: the momenta of a finite torus are a grid, and unless that grid
contains the Dirac point where the two bands meet, the gap of the finite sheet never closes.

The honeycomb lattice comes from the geometry API.  Its `bonds` are the nearest-neighbour
ones, so the second-neighbour bonds are generated from the three cell offsets separately, and
the mass from `colors('semenoff')`.  The patch is an $a\times a$ block of cells at the
corner, so it always includes part of the boundary.


In [ ]:
T, T2, PHI = 1.0, 0.2, math.pi / 2


def haldane_terms(geometry: Geometry, mass: float, t: float = T, t2: float = T2,
                  phi: float = PHI) -> list[tuple[int, int, complex]]:
    r'''
    Nearest-neighbour $t$, next-nearest $t_2e^{\pm i\varphi}$ and a Semenoff mass $M$.
    Args:
        geometry: the honeycomb sheet.
        mass: $M$, $+M$ on sublattice A and $-M$ on B.
        t: nearest-neighbour hopping.
        t2: next-nearest-neighbour hopping.
        phi: the flux, $+\varphi$ on sublattice A and $-\varphi$ on B.
    Returns:
        The (a, b, alpha) terms.
    '''
    terms = []
    for r, rp in geometry.bonds:
        terms += hopping(r, rp, complex(t))
    site_of = {(int(i), int(j), int(b)): r
               for r, ((i, j), b) in enumerate(zip(geometry.cell, geometry.basis))}

    def image(i: int, j: int) -> tuple[int, int] | None:
        '''The cell at an offset, wrapped where the sheet is periodic and dropped where it is not.'''
        if geometry.boundary[0] is Boundary.PERIODIC:
            i %= geometry.n1
        if geometry.boundary[1] is Boundary.PERIODIC:
            j %= geometry.n2
        return (i, j) if 0 <= i < geometry.n1 and 0 <= j < geometry.n2 else None

    # the three offsets that reach the other sublattice sites around a hexagon, each pair of
    # next-nearest neighbours met once because the opposite offsets are not walked
    for r, ((i, j), b) in enumerate(zip(geometry.cell, geometry.basis)):
        for di, dj in ((1, 0), (-1, 1), (0, -1)):
            cell = image(int(i) + di, int(j) + dj)
            if cell is not None:
                phase = (1.0 if b == 0 else -1.0) * phi
                terms += hopping(r, site_of[(cell[0], cell[1], int(b))],
                                 t2 * complex(math.cos(phase), math.sin(phase)))
    for r, xi in enumerate(1.0 - 2.0 * geometry.colors('semenoff').to(REAL)):
        terms.append((2 * r, 2 * r + 1, 0.5j * mass * float(xi)))
    return terms


CELLS, BLOCKS = 12, [1, 2, 3, 4, 5, 6, 7, 8]
SHEETS = {'topological, open': (0.0, Boundary.OPEN),
          'trivial, open': (1.5, Boundary.OPEN),
          'topological, torus': (0.0, Boundary.PERIODIC),
          'trivial, torus': (1.5, Boundary.PERIODIC)}
sheets, sheet_spectra = {}, {}
for label, (mass, boundary) in SHEETS.items():
    geometry = Geometry.build(HONEYCOMB, CELLS, CELLS, (boundary, boundary))
    matrix = majorana_matrix(haldane_terms(geometry, mass), 2 * geometry.sites)
    covariance = GfPEPS.from_hamiltonian(matrix).tensors[0]
    sheets[label] = {2 * a * a: patch_covariance(covariance, geometry.patch(a, a))
                     for a in BLOCKS}
    sheet_spectra[label] = positive_spectrum(matrix)

threshold = 3 * 3 ** 0.5 * T2 * math.sin(PHI)
torus = Geometry.build(HONEYCOMB, CELLS, CELLS, (Boundary.PERIODIC, Boundary.PERIODIC))
print(f'the torus gap closes at M = 3 sqrt(3) t2 sin(phi) = {threshold:.4f}:')
for mass in (0.9, 1.0, threshold, 1.1, 1.2):
    values = positive_spectrum(majorana_matrix(haldane_terms(torus, mass), 2 * torus.sites))
    print(f'  M = {mass:.4f}: gap = {float(values.min()):.4f}')

print(f'\n{"sheet":>20} {"min eps":>10} {"S(128)":>8} {"#mixed(128)":>12}')
for label, curves in sheets.items():
    print(f'{label:>20} {float(sheet_spectra[label].min()):>10.4f} '
          f'{entropy(curves[128]):>8.4f} {mixed_count(curves[128]):>12}')

figure, axes = plt.subplots(1, 3, figsize=(13.5, 3.4))
for label, values in sheet_spectra.items():
    axes[0].plot(range(1, len(values) + 1), values, '.', ms=3, label=label)
axes[0].set(xlabel='state index', ylabel=r'$\varepsilon_k$', yscale='log',
            ylim=(2e-2, 8), title='Haldane: single-particle spectrum')
for label, curves in sheets.items():
    sites = sorted(curves)
    axes[1].plot(sites, [mixed_count(curves[site]) for site in sites],
                 marker='.', label=label)
    axes[2].plot(sites, [entropy(curves[site]) for site in sites],
                 marker='.', label=label)
axes[1].set(xscale='log', yscale='log', xlabel=r'$\#$sites',
            ylabel=r'$\#\{\lambda_k < 1 - 10^{-8}\}$', title='Haldane: resolvable rank')
axes[2].set(xscale='log', xlabel=r'$\#$sites', ylabel='$S$', title='Haldane: entropy')
for axis in axes:
    axis.legend(fontsize=7)
figure.tight_layout()
plt.show()


## 4. The boundary of a patch, and how large the patch has to be

The blocks of section 3 fill much of their $12\times12$ sheet, and there the count still looks like
it follows the area.  The length to compare it with is the one the lattice measures itself,
`cut_bonds(patch)`: the bonds with one end inside the patch and one outside it.  Take the same four
sheets on a $30\times30$ lattice and grow the corner block from $a = 2$ to $14$ cells.

A mixed mode is a mode the cut leaves correlated with the outside, and that correlation reaches
only a finite depth into the patch, so the count is a layer along the cut and not a property of
the block: it grows with $a$ while the interior, which grows as $a^2$, comes out pure, and
`#mixed/sites` falls off like $1/a$.  Only the sides the block does not share with the sheet are
cut, so the open block cuts $2a$ bonds where the torus cuts $4a$: the count follows the cut, $1$
against $2$ at $a = 14$, and not the block, which is the same on both sheets.  `#mixed/cut` settles
at $4$ modes per bond on the open sheet, and the same law on the torus keeps a constant offset of
order $40$ modes from its four corners, so its ratio there has not quite settled at $a = 14$.  The first panel
is the block and the bonds it cuts, the second the count against its side with the two powers to
compare with, and the third the same count against the bonds it cuts.

In [ ]:
CELLS, SIDES = 30, [2, 4, 6, 8, 10, 12, 14]
big: dict[str, dict[int, tuple[int, Tensor]]] = {}
for label, (mass, boundary) in SHEETS.items():
    geometry = Geometry.build(HONEYCOMB, CELLS, CELLS, (boundary, boundary))
    matrix = majorana_matrix(haldane_terms(geometry, mass), 2 * geometry.sites)
    covariance = GfPEPS.from_hamiltonian(matrix).tensors[0]
    blocks = {}
    for a in SIDES:
        patch = geometry.patch(a, a)
        blocks[a] = (geometry.cut_bonds(patch), patch_covariance(covariance, patch))
    big[label] = blocks

print(f'the sheets of section 3, {CELLS} x {CELLS} cells, corner block of side a')
print(f'{"sheet":>20} {"a":>3} {"sites":>6} {"cut bonds":>10} {"#mixed":>7} {"S":>8} '
      f'{"#mixed/cut":>11} {"S/cut":>7} {"#mixed/sites":>12}')
for label, blocks in big.items():
    for a in SIDES:
        cut, block = blocks[a]
        count, value = mixed_count(block), entropy(block)
        print(f'{label:>20} {a:>3} {2 * a * a:>6} {cut:>10} {count:>7} {value:>8.4f} '
              f'{count / cut:>11.2f} {value / cut:>7.3f} {count / (2 * a * a):>12.3f}')
    print()

figure, axes = plt.subplots(1, 3, figsize=(14.4, 4.2))
geometry = Geometry.build(HONEYCOMB, CELLS, CELLS, (Boundary.OPEN, Boundary.OPEN))
positions = geometry.positions.cpu()
axes[0].scatter(positions[:, 0], positions[:, 1], s=1.5, color='0.82')
for a, colour in ((SIDES[-1], 'tab:blue'), (9, 'tab:orange'), (4, 'tab:green')):
    patch = geometry.patch(a, a)
    axes[0].scatter(positions[patch][:, 0], positions[patch][:, 1], s=4, color=colour,
                    label=rf'$a = {a}$: {2 * a * a} sites ({100 * 2 * a * a / (2 * CELLS ** 2):.0f}%)')
inside = set(geometry.patch(SIDES[-1], SIDES[-1]))
for r, rp in geometry.bonds:
    if (r in inside) != (rp in inside):
        axes[0].plot(positions[[r, rp], 0], positions[[r, rp], 1], '-', lw=1.4, color='tab:red')
axes[0].set_aspect('equal', adjustable='box')
axes[0].set(xlabel='$x$', ylabel='$y$', title='the corner block, and the bonds it cuts')
axes[0].legend(fontsize=7, markerscale=3)

for label, blocks in big.items():
    counts = [mixed_count(blocks[a][1]) for a in SIDES]
    axes[1].plot(SIDES, counts, marker='.', label=label)
    axes[2].plot([blocks[a][0] for a in SIDES], counts, marker='.', label=label)
axes[1].plot(SIDES, [16 * a for a in SIDES], 'k--', lw=1, label=r'$4$ modes per cut bond')
axes[1].plot(SIDES, [2 * a * a for a in SIDES], 'k:', lw=1, label=r'every mode of the block')
axes[1].set(xscale='log', yscale='log', xlabel='$a$', ylabel=r'$\#$mixed',
            title='against the side of the block')
axes[1].legend(fontsize=7)
axes[2].plot([4 * a for a in SIDES], [16 * a for a in SIDES], 'k--', lw=1,
             label=r'$4$ modes per bond')
axes[2].set(xlabel='cut bonds of the patch', ylabel=r'$\#$mixed',
            title='against the bonds it cuts')
axes[2].legend(fontsize=7)
figure.tight_layout()
plt.show()

## 5. The same boundary at every tolerance

$\epsilon = 10^{-8}$ is not a special value: at every tolerance the count in units of the cut bonds
is flat in $a$ once the block is wide enough for the layer at the cut to be a small part of it, so
the linear growth of section 4 is not an artifact of where the tolerance was put.  The tolerance
sets the *level* of that ratio and moves it slowly, with $\log(1/\epsilon)$: $1.2$ modes per cut
bond at $10^{-2}$ against $5.1$ at $10^{-12}$ on the torus, since a tighter tolerance resolves a
deeper layer at the cut.  The proportionality to the boundary is what does not change.

In [ ]:
print('the count in units of the cut bonds, at every tolerance')
print(f'{"sheet":>20} {"a":>3} {"cut":>4} '
      + ' '.join(f'{tolerance:>7.0e}' for tolerance in TOLERANCES))
for label in ('topological, open', 'topological, torus'):
    for a in SIDES:
        cut, block = big[label][a]
        p = occupations(block)
        print(f'{label:>20} {a:>3} {cut:>4} '
              + ' '.join(f'{int((p > tolerance / 2).sum()) / cut:>7.2f}'
                         for tolerance in TOLERANCES))
    print()

figure, axes = plt.subplots(1, 2, figsize=(10.0, 3.6))
for axis, label in zip(axes, ('topological, open', 'topological, torus')):
    for tolerance in TOLERANCES:
        axis.plot(SIDES,
                  [int((occupations(big[label][a][1]) > tolerance / 2).sum()) / big[label][a][0]
                   for a in SIDES],
                  marker='.', ms=4, label=rf'${tolerance:.0e}$')
    axis.set(xlabel='$a$', ylabel=r'$\#$mixed / cut bonds', title=label, ylim=(0, 6.0))
    axis.legend(fontsize=6, ncol=3, loc='lower right')
figure.tight_layout()
plt.show()